# 03 Descriptive Statistics (Learn-The-Data)

Learn-the-data version: this notebook teaches the processed panel from three angles:

1) Data overview (coverage + missingness)
2) Univariate structure (per-variable distributions + time trends)
3) Bivariate + multivariate structure (correlations, scatter sweeps, collinearity signals)

The notebook automatically detects **all numeric columns** in `data/processed/clean_panel.csv` (excluding IDs)
and saves key figures + CSV tables under `outputs/`.


## Setup


In [15]:
from pathlib import Path
import sys
import importlib

import numpy as np
import pandas as pd
from scipy.stats import gaussian_kde

import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import src.config as config
importlib.reload(config)

OUTPUTS_DIR = config.OUTPUTS_DIR
FIGURES_DIR = config.FIGURES_DIR
PROCESSED_PANEL_FILE = config.PROCESSED_PANEL_FILE
TIME_WINDOW = config.TIME_WINDOW
ensure_output_dirs = config.ensure_output_dirs

ensure_output_dirs()
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


## Load Panel (Processed Contract)

Source of truth: `data/processed/clean_panel.csv` produced by `01_data_processing.ipynb`.


In [16]:
df = pd.read_csv(PROCESSED_PANEL_FILE)
df.shape, int(df['year'].min()), int(df['year'].max())


((264, 18), 2000, 2023)

In [17]:
start_year, end_year = TIME_WINDOW
year_min, year_max = int(df['year'].min()), int(df['year'].max())
if (year_min, year_max) != (start_year, end_year):
    raise ValueError(f'Expected year window {TIME_WINDOW}, got {(year_min, year_max)}. Re-run 01_data_processing.ipynb.')

exclude = {'country_id', 'year', 'country', 'country_code'}
numeric_cols = [
    c for c in df.select_dtypes(include=[np.number]).columns
    if c not in exclude
]
if not numeric_cols:
    raise ValueError('No numeric columns detected after exclusions; check processed panel schema.')

display(df[['country', 'country_code', 'year'] + [c for c in ['fdi_pct_gdp', 'broad_money_growth_pct'] if c in df.columns]].head())
numeric_cols[:10], len(numeric_cols)


,country,country_code,year,fdi_pct_gdp,broad_money_growth_pct
0,Brunei Darussalam,BRN,2000,8.3641,47.6608
1,Brunei Darussalam,BRN,2001,0.9956,-11.9304
2,Brunei Darussalam,BRN,2002,3.6265,1.8854
3,Brunei Darussalam,BRN,2003,1.7275,4.0711
4,Brunei Darussalam,BRN,2004,1.3134,15.8347


(['fdi_pct_gdp',
  'fdi_pct_gdp_winsorized',
  'broad_money_growth_pct',
  'trade_pct_gdp',
  'inflation_gdp_deflator_pct',
  'deposit_interest_rate_pct',
  'real_interest_rate_pct',
  'lending_interest_rate_pct',
  'hc_human_capital_index',
  'ln_gdppc'],
 14)

## 1) Data Overview (Sanity + Coverage)

Deliverables:
- `outputs/descriptive_numeric_columns.csv`
- `outputs/missingness_by_variable.csv`
- `outputs/coverage_by_country_numeric.csv`


In [18]:
numeric_columns_df = pd.DataFrame({'numeric_column': numeric_cols})
numeric_columns_path = OUTPUTS_DIR / 'descriptive_numeric_columns.csv'
numeric_columns_df.to_csv(numeric_columns_path, index=False)
print('Saved', numeric_columns_path)

missingness = pd.DataFrame({
    'variable': numeric_cols,
    'missing_count': [int(df[c].isna().sum()) for c in numeric_cols],
})
missingness['missing_rate'] = missingness['missing_count'] / len(df)
missingness = missingness.sort_values(['missing_rate', 'variable'], ascending=[False, True]).reset_index(drop=True)
missingness_path = OUTPUTS_DIR / 'missingness_by_variable.csv'
missingness.to_csv(missingness_path, index=False)
print('Saved', missingness_path)

coverage_by_country = (
    df.groupby('country')[numeric_cols]
      .apply(lambda frame: frame.notna().sum())
      .reset_index()
)
coverage_by_country_path = OUTPUTS_DIR / 'coverage_by_country_numeric.csv'
coverage_by_country.to_csv(coverage_by_country_path, index=False)
print('Saved', coverage_by_country_path)

missingness.head(20)


Saved /Users/bunnypro/Projects/monetary_policy_fdi_analysis/outputs/descriptive_numeric_columns.csv
Saved /Users/bunnypro/Projects/monetary_policy_fdi_analysis/outputs/missingness_by_variable.csv
Saved /Users/bunnypro/Projects/monetary_policy_fdi_analysis/outputs/coverage_by_country_numeric.csv


,variable,missing_count,missing_rate
0,lending_interest_rate_pct,72,0.2727
1,real_interest_rate_pct,59,0.2235
2,deposit_interest_rate_pct,51,0.1932
3,broad_money_growth_pct,24,0.0909
4,hc_human_capital_index,24,0.0909
5,ln_population_total,24,0.0909
6,ln_tourism_arrivals,24,0.0909
7,trade_pct_gdp,24,0.0909
8,xr_dep_pct,11,0.0417
9,xr_dep_pct_winsorized,11,0.0417


## 2) Univariate Analysis (Per Variable)

For each numeric column `x`, produce:
- Summary stats in `outputs/univariate_summary.csv`
- Distribution plots in `outputs/figures/univariate_dist__{col}.png`
- Time-trend plots in `outputs/figures/univariate_trend__{col}.png`


In [19]:
def safe_slug(name: str) -> str:
    return ''.join(ch if (ch.isalnum() or ch in {'_', '-'}) else '_' for ch in name).strip('_')

def compute_univariate_summary(series: pd.Series) -> dict:
    values = series.dropna().astype(float)
    missing = int(series.isna().sum())
    missing_rate = missing / len(series)
    if values.empty:
        return {
            'count': 0,
            'mean': np.nan,
            'std': np.nan,
            'min': np.nan,
            'p25': np.nan,
            'median': np.nan,
            'p75': np.nan,
            'max': np.nan,
            'missing': missing,
            'missing_pct': 100 * missing_rate,
        }
    q = values.quantile([0.25, 0.5, 0.75]).to_dict()
    return {
        'count': int(values.shape[0]),
        'mean': float(values.mean()),
        'std': float(values.std(ddof=1)),
        'min': float(values.min()),
        'p25': float(q.get(0.25, np.nan)),
        'median': float(q.get(0.5, np.nan)),
        'p75': float(q.get(0.75, np.nan)),
        'max': float(values.max()),
        'missing': missing,
        'missing_pct': 100 * missing_rate,
    }

def plot_distribution(series: pd.Series, col: str) -> None:
    values = series.dropna().astype(float).values
    slug = safe_slug(col)
    out_path = FIGURES_DIR / f'univariate_dist__{slug}.png'

    fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
    ax_hist, ax_box = axes

    if values.size == 0:
        ax_hist.text(0.5, 0.5, 'No non-missing values', ha='center', va='center')
        ax_box.text(0.5, 0.5, 'No non-missing values', ha='center', va='center')
    else:
        ax_hist.hist(values, bins=30, density=True, alpha=0.6, color='steelblue', edgecolor='white')
        if np.unique(values).size >= 5:
            xs = np.linspace(np.nanmin(values), np.nanmax(values), 200)
            kde = gaussian_kde(values)
            ax_hist.plot(xs, kde(xs), color='black', linewidth=1.2, label='KDE')
            ax_hist.legend(loc='best', frameon=False)
        ax_hist.set_title(f'Histogram + KDE: {col}')
        ax_hist.set_xlabel(col)
        ax_hist.set_ylabel('Density')

        ax_box.boxplot(values, vert=False, showfliers=True)
        ax_box.set_title('Boxplot (outliers)')
        ax_box.set_xlabel(col)

    fig.tight_layout()
    fig.savefig(out_path, dpi=200)
    plt.close(fig)

def plot_time_trend(frame: pd.DataFrame, col: str) -> None:
    slug = safe_slug(col)
    out_path = FIGURES_DIR / f'univariate_trend__{slug}.png'

    tmp = frame[['year', col]].dropna()
    if tmp.empty:
        fig, ax = plt.subplots(figsize=(10, 3.6))
        ax.text(0.5, 0.5, f'No non-missing values for {col}', ha='center', va='center')
        ax.set_axis_off()
        fig.tight_layout()
        fig.savefig(out_path, dpi=200)
        plt.close(fig)
        return

    grouped = tmp.groupby('year')[col]
    summary = pd.DataFrame({
        'mean': grouped.mean(),
        'median': grouped.median(),
        'n': grouped.size(),
        'p25': grouped.quantile(0.25),
        'p75': grouped.quantile(0.75),
    }).reset_index()

    fig, ax = plt.subplots(figsize=(10, 3.8))
    ax.plot(summary['year'], summary['mean'], label='Mean', color='steelblue')
    ax.plot(summary['year'], summary['median'], label='Median', color='darkorange', linewidth=1.2)

    ok_band = summary['n'] >= 10
    if ok_band.any():
        ax.fill_between(
            summary.loc[ok_band, 'year'],
            summary.loc[ok_band, 'p25'],
            summary.loc[ok_band, 'p75'],
            color='gray',
            alpha=0.15,
            label='IQR band (n>=10)',
        )

    ax.set_title(f'Panel-average time trend: {col}')
    ax.set_xlabel('Year')
    ax.legend(loc='best', frameon=False)
    fig.tight_layout()
    fig.savefig(out_path, dpi=200)
    plt.close(fig)

rows = []
for col in numeric_cols:
    summary = compute_univariate_summary(df[col])
    rows.append({'variable': col, **summary})
    plot_distribution(df[col], col)
    plot_time_trend(df, col)

univariate_summary = pd.DataFrame(rows).sort_values('variable').reset_index(drop=True)
univariate_summary_path = OUTPUTS_DIR / 'univariate_summary.csv'
univariate_summary.to_csv(univariate_summary_path, index=False)
print('Saved', univariate_summary_path)
univariate_summary.head(15)


Saved /Users/bunnypro/Projects/monetary_policy_fdi_analysis/outputs/univariate_summary.csv


,variable,count,mean,std,min,p25,median,p75,max,missing,missing_pct
0,broad_money_growth_pct,240,14.3104,11.8193,-11.9304,6.1694,10.5745,19.9642,61.8365,24,9.0909
1,deposit_interest_rate_pct,213,4.0804,3.5598,0.1217,1.3380,3.0481,6.4475,15.5025,51,19.3182
2,fdi_pct_gdp,260,4.9183,6.1702,-16.2053,1.8826,3.2578,5.3786,32.5965,4,1.5152
3,fdi_pct_gdp_winsorized,260,4.9637,5.9226,-4.3303,1.8826,3.2578,5.3786,28.2639,4,1.5152
4,hc_human_capital_index,240,2.3835,0.4758,1.5157,1.9112,2.4808,2.7215,3.3993,24,9.0909
5,inflation_gdp_deflator_pct,264,5.2976,8.2354,-21.7393,1.7465,3.8869,7.3097,59.0797,0,0.0000
6,lending_interest_rate_pct,192,9.7832,6.3922,3.0600,5.3800,6.8116,12.6194,32.0000,72,27.2727
7,ln_gdppc,264,24.7187,2.0186,19.7207,23.3028,25.2052,26.4523,27.9467,0,0.0000
8,ln_population_total,240,16.9487,1.7974,12.6960,15.6867,17.5015,18.2563,19.4545,24,9.0909
9,ln_tourism_arrivals,240,15.2576,1.1497,12.9384,14.3817,15.2620,16.1335,17.5023,24,9.0909


### Robustness Variant Checks (Winsorized vs Raw)

When both raw and winsorized variants exist for the same base variable, compare their distributions.


In [20]:
def find_winsor_pairs(columns: list[str]) -> list[tuple[str, str]]:
    pairs = []
    for col in columns:
        if col.endswith('_winsorized'):
            base = col[:-len('_winsorized')]
            if base in columns:
                pairs.append((base, col))
    return sorted(pairs)

winsor_pairs = find_winsor_pairs(numeric_cols)
winsor_pairs[:10], len(winsor_pairs)


([('fdi_pct_gdp', 'fdi_pct_gdp_winsorized'),
  ('xr_dep_pct', 'xr_dep_pct_winsorized')],
 2)

In [21]:
def compare_winsor_pair(raw_col: str, wins_col: str) -> None:
    raw = df[raw_col].dropna().astype(float)
    win = df[wins_col].dropna().astype(float)
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
    for ax, series, title in [
        (axes[0], raw, f'Raw: {raw_col}'),
        (axes[1], win, f'Winsorized: {wins_col}'),
    ]:
        if series.empty:
            ax.text(0.5, 0.5, 'No values', ha='center', va='center')
            continue
        ax.hist(series.values, bins=30, density=True, alpha=0.6, color='steelblue', edgecolor='white')
        if np.unique(series.values).size >= 5:
            xs = np.linspace(series.min(), series.max(), 200)
            kde = gaussian_kde(series.values)
            ax.plot(xs, kde(xs), color='black', linewidth=1.2)
        ax.set_title(title)
    fig.tight_layout()
    out_path = FIGURES_DIR / f'univariate_dist_compare__{safe_slug(raw_col)}.png'
    fig.savefig(out_path, dpi=200)
    plt.close(fig)

for raw_col, wins_col in winsor_pairs:
    compare_winsor_pair(raw_col, wins_col)

print(f'Generated {len(winsor_pairs)} winsorized vs raw comparison plots (if any).')


Generated 2 winsorized vs raw comparison plots (if any).


## 3) Bivariate Analysis (Pairs)

Deliverables:
- `outputs/correlation_matrix_numeric.csv`
- `outputs/correlation_pairwise_n_numeric.csv`
- `outputs/figures/bivariate_fdi_vs__{col}.png`
- `outputs/figures/bivariate_bm_growth_vs__{col}.png`
- `outputs/top_correlations_with_fdi.csv`


In [22]:
numeric_df = df[numeric_cols].copy()

corr = numeric_df.corr()
corr_path = OUTPUTS_DIR / 'correlation_matrix_numeric.csv'
corr.to_csv(corr_path)
print('Saved', corr_path)

pairwise_n = pd.DataFrame(index=numeric_cols, columns=numeric_cols, dtype=int)
for i, a in enumerate(numeric_cols):
    a_vals = numeric_df[a]
    for b in numeric_cols[i:]:
        n = int(pd.concat([a_vals, numeric_df[b]], axis=1).dropna().shape[0])
        pairwise_n.loc[a, b] = n
        pairwise_n.loc[b, a] = n

pairwise_n_path = OUTPUTS_DIR / 'correlation_pairwise_n_numeric.csv'
pairwise_n.to_csv(pairwise_n_path)
print('Saved', pairwise_n_path)

corr.iloc[:8, :8]


Saved /Users/bunnypro/Projects/monetary_policy_fdi_analysis/outputs/correlation_matrix_numeric.csv
Saved /Users/bunnypro/Projects/monetary_policy_fdi_analysis/outputs/correlation_pairwise_n_numeric.csv


,fdi_pct_gdp,fdi_pct_gdp_winsorized,broad_money_growth_pct,trade_pct_gdp,inflation_gdp_deflator_pct,deposit_interest_rate_pct,real_interest_rate_pct,lending_interest_rate_pct
fdi_pct_gdp,1.0000,0.9914,-0.0071,0.7787,-0.0994,-0.3721,-0.0993,-0.2243
fdi_pct_gdp_winsorized,0.9914,1.0000,-0.0119,0.7963,-0.1187,-0.3741,-0.0635,-0.2253
broad_money_growth_pct,-0.0071,-0.0119,1.0000,-0.1281,0.3171,0.4334,0.0336,0.6236
trade_pct_gdp,0.7787,0.7963,-0.1281,1.0000,-0.1252,-0.4250,-0.1298,-0.3571
inflation_gdp_deflator_pct,-0.0994,-0.1187,0.3171,-0.1252,1.0000,0.5137,-0.7355,0.4548
deposit_interest_rate_pct,-0.3721,-0.3741,0.4334,-0.4250,0.5137,1.0000,0.0650,0.7351
real_interest_rate_pct,-0.0993,-0.0635,0.0336,-0.1298,-0.7355,0.0650,1.0000,0.3573
lending_interest_rate_pct,-0.2243,-0.2253,0.6236,-0.3571,0.4548,0.7351,0.3573,1.0000


In [23]:
def choose_fdi_column(columns: list[str]) -> str:
    if 'fdi_pct_gdp_winsorized' in columns:
        return 'fdi_pct_gdp_winsorized'
    if 'fdi_pct_gdp' in columns:
        return 'fdi_pct_gdp'
    raise ValueError('Could not find FDI column: expected fdi_pct_gdp(_winsorized).')

fdi_col = choose_fdi_column(df.columns.tolist())
fdi_col


'fdi_pct_gdp_winsorized'

In [24]:
def scatter_with_fit(x: pd.Series, y: pd.Series, x_name: str, y_name: str, out_path: Path) -> None:
    tmp = pd.concat([x.rename(x_name), y.rename(y_name)], axis=1).dropna()
    n = int(tmp.shape[0])
    corr_xy = float(tmp[x_name].corr(tmp[y_name])) if n >= 2 else np.nan

    fig, ax = plt.subplots(figsize=(6.5, 4.2))
    ax.scatter(tmp[x_name], tmp[y_name], alpha=0.35, s=14, color='steelblue')

    if n >= 2 and np.isfinite(tmp[x_name]).all() and np.isfinite(tmp[y_name]).all():
        slope, intercept = np.polyfit(tmp[x_name], tmp[y_name], 1)
        xs = np.linspace(tmp[x_name].min(), tmp[x_name].max(), 100)
        ax.plot(xs, intercept + slope * xs, color='darkorange', linewidth=1.8, label='OLS fit')
        ax.legend(loc='best', frameon=False)

    ax.set_xlabel(x_name)
    ax.set_ylabel(y_name)
    ax.set_title(f'{y_name} vs {x_name}')
    ax.text(
        0.02,
        0.98,
        f'corr={corr_xy:,.3f}\nN={n}',
        transform=ax.transAxes,
        ha='left',
        va='top',
        bbox={'facecolor': 'white', 'alpha': 0.75, 'edgecolor': 'none'},
    )
    fig.tight_layout()
    fig.savefig(out_path, dpi=200)
    plt.close(fig)

# FDI vs X sweep
for col in numeric_cols:
    if col == fdi_col or col.startswith('fdi'):
        continue
    out_path = FIGURES_DIR / f'bivariate_fdi_vs__{safe_slug(col)}.png'
    scatter_with_fit(df[fdi_col], df[col], fdi_col, col, out_path)

# Broad money growth vs X sweep (if present)
if 'broad_money_growth_pct' in df.columns:
    bm_col = 'broad_money_growth_pct'
    for col in numeric_cols:
        if col == bm_col:
            continue
        out_path = FIGURES_DIR / f'bivariate_bm_growth_vs__{safe_slug(col)}.png'
        scatter_with_fit(df[bm_col], df[col], bm_col, col, out_path)
else:
    print('broad_money_growth_pct not found; skipping BM-growth sweep.')

print('Bivariate sweeps complete.')


Bivariate sweeps complete.


In [25]:
# Top correlations with FDI (exclude mechanically related FDI columns)
fdi_corr = corr[fdi_col].drop(index=[c for c in corr.index if c.startswith('fdi')], errors='ignore')
fdi_corr = fdi_corr.dropna()
top_k = 12
top_pos = fdi_corr.sort_values(ascending=False).head(top_k)
top_neg = fdi_corr.sort_values(ascending=True).head(top_k)
top_table = (
    pd.concat(
        [
            top_pos.rename('corr').to_frame().assign(direction='positive'),
            top_neg.rename('corr').to_frame().assign(direction='negative'),
        ]
    )
    .reset_index()
    .rename(columns={'index': 'variable'})
)
top_table['pairwise_n'] = top_table['variable'].map(lambda v: int(pairwise_n.loc[fdi_col, v]) if v in pairwise_n.columns else np.nan)
top_out_path = OUTPUTS_DIR / 'top_correlations_with_fdi.csv'
top_table.to_csv(top_out_path, index=False)
print('Saved', top_out_path)
top_table


Saved /Users/bunnypro/Projects/monetary_policy_fdi_analysis/outputs/top_correlations_with_fdi.csv


,variable,corr,direction,pairwise_n
0,trade_pct_gdp,0.7963,positive,236
1,hc_human_capital_index,0.2700,positive,240
2,ln_tourism_arrivals,0.1647,positive,240
3,ln_gdppc,0.1638,positive,260
4,broad_money_growth_pct,-0.0119,positive,239
5,xr_dep_pct,-0.0403,positive,250
6,real_interest_rate_pct,-0.0635,positive,205
7,xr_dep_pct_winsorized,-0.0679,positive,250
8,inflation_gdp_deflator_pct,-0.1187,positive,260
9,lending_interest_rate_pct,-0.2253,positive,192


## 4) Multivariate Analysis (Correlation Structure)

Deliverables:
- `outputs/figures/correlation_heatmap_numeric.png`
- `outputs/high_correlation_pairs.csv` (|corr| >= 0.7 by default)
- `outputs/vif_learning_check.csv` (if columns available)


In [26]:
# Heatmap
n = len(numeric_cols)
fig_w = max(10, min(24, 0.55 * n))
fig_h = max(8, min(22, 0.55 * n))
fig, ax = plt.subplots(figsize=(fig_w, fig_h))
im = ax.imshow(corr.values, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(numeric_cols, rotation=90, fontsize=7)
ax.set_yticklabels(numeric_cols, fontsize=7)
ax.set_title('Correlation heatmap (numeric columns)')
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
heatmap_path = FIGURES_DIR / 'correlation_heatmap_numeric.png'
fig.savefig(heatmap_path, dpi=220)
plt.close(fig)
print('Saved', heatmap_path)


Saved /Users/bunnypro/Projects/monetary_policy_fdi_analysis/outputs/figures/correlation_heatmap_numeric.png


In [27]:
# High-correlation edges table
threshold = 0.7
pairs = []
for i, a in enumerate(numeric_cols):
    for j in range(i + 1, len(numeric_cols)):
        b = numeric_cols[j]
        value = corr.loc[a, b]
        if pd.isna(value):
            continue
        if abs(value) >= threshold:
            pairs.append(
                {
                    'var_a': a,
                    'var_b': b,
                    'corr': float(value),
                    'abs_corr': float(abs(value)),
                    'pairwise_n': int(pairwise_n.loc[a, b]),
                }
            )
high_corr_pairs = pd.DataFrame(pairs).sort_values(['abs_corr', 'pairwise_n'], ascending=[False, False]).reset_index(drop=True)
high_corr_path = OUTPUTS_DIR / 'high_correlation_pairs.csv'
high_corr_pairs.to_csv(high_corr_path, index=False)
print('Saved', high_corr_path, f'(threshold={threshold})')
high_corr_pairs.head(25)


Saved /Users/bunnypro/Projects/monetary_policy_fdi_analysis/outputs/high_correlation_pairs.csv (threshold=0.7)


,var_a,var_b,corr,abs_corr,pairwise_n
0,fdi_pct_gdp,fdi_pct_gdp_winsorized,0.9914,0.9914,260
1,lending_interest_rate_pct,hc_human_capital_index,-0.8487,0.8487,192
2,fdi_pct_gdp_winsorized,trade_pct_gdp,0.7963,0.7963,236
3,fdi_pct_gdp,trade_pct_gdp,0.7787,0.7787,236
4,inflation_gdp_deflator_pct,real_interest_rate_pct,-0.7355,0.7355,205
5,deposit_interest_rate_pct,lending_interest_rate_pct,0.7351,0.7351,189
6,ln_gdppc,ln_tourism_arrivals,0.7228,0.7228,240


In [28]:
# Collinearity quick check: VIF on workbook-core controls (learning-oriented)
try:
    from statsmodels.stats.outliers_influence import variance_inflation_factor
    from statsmodels.tools.tools import add_constant
except Exception as exc:
    variance_inflation_factor = None
    print('statsmodels not available; cannot compute VIF:', exc)

core_controls = [
    'broad_money_growth_pct',
    'inflation_gdp_deflator_pct',
    'trade_pct_gdp',
    'ln_gdppc',
    'xr_dep_pct',
]
rate_proxy_candidates = ['real_interest_rate_pct', 'deposit_interest_rate_pct', 'lending_interest_rate_pct']
rate_proxy = next((c for c in rate_proxy_candidates if c in df.columns), None)
if rate_proxy is not None and rate_proxy not in core_controls:
    core_controls = core_controls + [rate_proxy]

missing = [c for c in core_controls if c not in df.columns]
if missing:
    print('Skipping VIF: missing required columns:', missing)
elif variance_inflation_factor is None:
    print('Skipping VIF: statsmodels missing.')
else:
    X = df[core_controls].dropna().astype(float)
    if X.shape[0] < 10:
        print(f'Skipping VIF: too few complete rows for VIF (n={X.shape[0]}).')
    else:
        Xc = add_constant(X, has_constant='add')
        vif_rows = []
        for i, col in enumerate(Xc.columns):
            if col == 'const':
                continue
            vif_rows.append({'variable': col, 'vif': float(variance_inflation_factor(Xc.values, i)), 'n': int(X.shape[0])})
        vif_df = pd.DataFrame(vif_rows).sort_values('vif', ascending=False).reset_index(drop=True)
        vif_path = OUTPUTS_DIR / 'vif_learning_check.csv'
        vif_df.to_csv(vif_path, index=False)
        print('Saved', vif_path)
        vif_df


Saved /Users/bunnypro/Projects/monetary_policy_fdi_analysis/outputs/vif_learning_check.csv
